# Breast Cancer Survival Analysis
## Notebook 2 — Statistical Hypothesis Testing

This notebook tests three hypotheses derived from the patterns observed in Notebook 1. Each test uses a regression-based approach and follows the same structure: model definition, null and alternative hypotheses, test statistic, and a plain-English interpretation of the result.

**Significance level:** α = 0.05 across all tests.

**Prerequisite:** Run `01_eda.ipynb` first to understand the data before interpreting these results.

---

### Contents
1. [Setup & Data Preparation](#1-setup--data-preparation)
2. [H1 — Regional Nodes x Population on Survival Months](#2-h1--regional-nodes-x-population-on-survival-months)
3. [H2 — Estrogen Status x Population on Survival Outcome](#3-h2--estrogen-status-x-population-on-survival-outcome)
4. [H3 — Continuous vs Binary Tumour Size Encoding](#4-h3--continuous-vs-binary-tumour-size-encoding)
5. [Summary of Results](#5-summary-of-results)

---
## 1. Setup & Data Preparation

We load both cohorts and combine them into a single DataFrame with a binary `Population` indicator (0 = SEER, 1 = METABRIC). The interaction terms in H1 and H2 rely on this indicator to detect population-level differences.

In [1]:
# ── Working directory ──────────────────────────────────────────────────────────
# Notebooks run relative to where Jupyter was launched, not where the notebook
# lives. This moves to the project root so data/ paths resolve correctly.
import os
try:
    notebook_dir = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    notebook_dir = os.path.abspath('')
os.chdir(notebook_dir)
#print('Working directory:', os.getcwd())

import math
import warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# ── Config ─────────────────────────────────────────────────────────────────────
DATA_PATH = 'data/clean_data_breast_cancer.xlsx'
ALPHA     = 0.05

COLUMN_RENAME = {
    'Survival Months': 'Survival_Months', 'Regional Node 1': 'Regional_Node',
    'Estrogen Status': 'Estrogen', 'Progesterone Status': 'Progesterone',
    'Tumor Size': 'Tumor_Size', 'T Stage': 'T_Stage',
}

In [2]:
# ── Helpers ────────────────────────────────────────────────────────────────────
def load_and_combine(path):
    seer     = pd.read_excel(path, sheet_name='SEER')
    metabric = pd.read_excel(path, sheet_name='METABRIC')
    seer['Population'], metabric['Population'] = 0, 1
    combined = pd.concat([seer, metabric], ignore_index=True)
    return combined.rename(columns=COLUMN_RENAME)

def print_section(title):
    print(f"\n{'='*60}\n  {title}\n{'='*60}\n")

def print_test_result(term, beta, se, t0, pv):
    decision = f'Reject H0 (p = {pv:.4e})' if pv < ALPHA else f'Fail to reject H0 (p = {pv:.4e})'
    print(f'  Term tested     : {term}')
    print(f'  Coefficient     : {beta:.4f}')
    print(f'  Std. error      : {se:.4f}')
    print(f'  t-statistic     : {t0:.4f}')
    print(f'  Decision (a={ALPHA}) : {decision}\n')

def run_ttest(result, term, n):
    """Extract coefficient and SE from a fitted model and run a two-sided t-test."""
    beta = result.params[term]
    se   = result.bse[term]
    t0   = beta / (se / math.sqrt(n))
    pv   = 2 * min(stats.t.cdf(t0, df=n-1), 1 - stats.t.cdf(t0, df=n-1))
    return beta, se, t0, pv

In [3]:
data = load_and_combine(DATA_PATH)
print(f'Combined dataset: {data.shape[0]:,} rows x {data.shape[1]} columns')
data.head()

Combined dataset: 5,359 rows x 10 columns


,Age,T_Stage,Grade,Tumor_Size,Estrogen,Progesterone,Regional_Node,Survival_Months,Status,Population
0,43.0,2,2,40.0,1,1,11,1.0,1,0
1,47.0,2,2,45.0,1,1,9,2.0,1,0
2,67.0,2,3,25.0,1,1,1,2.0,0,0
3,46.0,1,2,19.0,1,1,1,2.0,0,0
4,63.0,2,2,35.0,1,1,5,3.0,0,0


---
## 2. H1 — Regional Nodes x Population on Survival Months

### Motivation
In both correlation matrices (Notebook 1), Regional Node was among the strongest correlates of Survival Months. However, the two cohorts differ substantially in their node distributions — SEER patients have more nodes on average (mean 4.15 vs 1.83 in METABRIC). This raises the question of whether the *penalty* of each additional node on survival time is the same across cohorts.

### Model
An OLS regression with an interaction term between Regional Node count and Population indicator:

$$\text{Survival\_Months} = \beta_0 + \beta_{RN} \cdot \text{Regional\_Node} + \beta_{Pop} \cdot \text{Population} + \beta_{RN \times Pop} \cdot \text{Regional\_Node} \times \text{Population}$$

The coefficient of interest is $\beta_{RN \times Pop}$: if it is significantly different from zero, the per-node effect on survival months is population-dependent.

### Hypotheses
- **H0:** $\beta_{RN \times Pop} = 0$ — the per-node penalty is the same in both cohorts
- **H1:** $\beta_{RN \times Pop} \neq 0$

In [4]:
print_section('Hypothesis 1 — OLS: Survival Months ~ Population x Regional Nodes')

model_h1  = smf.ols('Survival_Months ~ Regional_Node * Population', data=data)
result_h1 = model_h1.fit()
print(result_h1.summary())

beta, se, t0, pv = run_ttest(result_h1, 'Regional_Node:Population', len(data))
print_test_result('Regional_Node:Population', beta, se, t0, pv)


  Hypothesis 1 — OLS: Survival Months ~ Population x Regional Nodes

                            OLS Regression Results                            
Dep. Variable:        Survival_Months   R-squared:                       0.277
Model:                            OLS   Adj. R-squared:                  0.277
Method:                 Least Squares   F-statistic:                     685.2
Date:                Wed, 13 May 2026   Prob (F-statistic):               0.00
Time:                        22:44:01   Log-Likelihood:                -27715.
No. Observations:                5359   AIC:                         5.544e+04
Df Residuals:                    5355   BIC:                         5.546e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------

### Interpretation

We **reject H0**. The interaction term is statistically significant (p < 0.05), meaning the effect of each additional positive regional node on survival months is not the same across cohorts.

The negative interaction coefficient tells us that each additional node in METABRIC reduces expected survival by approximately 4.5 months *more* than the equivalent node in SEER. This may reflect differences in clinical protocols, patient demographics, or the longer follow-up window in METABRIC exposing more late-stage deterioration.

> **Implication for modelling:** Population should be included as a covariate in any survival model that combines both cohorts.

---
## 3. H2 — Estrogen Status x Population on Survival Outcome

### Motivation
From the EDA, estrogen receptor positivity rates differ substantially between cohorts (93% in SEER vs 77% in METABRIC). Positive estrogen receptors allow patients to receive hormone therapy, which can improve prognosis. We want to test whether this survival benefit is consistent across populations, or whether it interacts with which cohort a patient belongs to.

### Model
A logistic regression with an interaction term between Estrogen Status and Population indicator:

$$\text{logit}(P(\text{Status})) = \beta_0 + \beta_{Pop} \cdot \text{Population} + \beta_{Estr} \cdot \text{Estrogen} + \beta_{Pop \times Estr} \cdot \text{Population} \times \text{Estrogen}$$

The coefficient of interest is $\beta_{Pop \times Estr}$: if significant, the estrogen effect on survival odds differs by population.

### Hypotheses
- **H0:** $\beta_{Pop \times Estr} = 0$ — estrogen affects survival equally in both cohorts
- **H1:** $\beta_{Pop \times Estr} \neq 0$

In [5]:
print_section('Hypothesis 2 — Logit: Status ~ Population x Estrogen Status')

model_h2  = smf.logit('Status ~ Population * Estrogen', data=data)
result_h2 = model_h2.fit()
print(result_h2.summary())

beta, se, t0, pv = run_ttest(result_h2, 'Population:Estrogen', len(data))
print_test_result('Population:Estrogen', beta, se, t0, pv)


  Hypothesis 2 — Logit: Status ~ Population x Estrogen Status

Optimization terminated successfully.
         Current function value: 0.481892
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                 Status   No. Observations:                 5359
Model:                          Logit   Df Residuals:                     5355
Method:                           MLE   Df Model:                            3
Date:                Wed, 13 May 2026   Pseudo R-squ.:                  0.1508
Time:                        22:44:07   Log-Likelihood:                -2582.5
converged:                       True   LL-Null:                       -3041.0
Covariance Type:            nonrobust   LLR p-value:                1.779e-198
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept               0.

### Interpretation

We **reject H0**. The interaction term is statistically significant (p < 0.05), meaning the survival benefit of positive estrogen receptors is not consistent across populations.

This could reflect differences in how hormone therapy is administered across the two clinical settings, or differences in the patient populations that happen to have positive receptors. The result reinforces that Population is a meaningful moderating variable in this dataset — not just a label.

> **Implication for modelling:** Estrogen status and Population should not be treated as independent additive effects in a combined-cohort model.

---
## 4. H3 — Continuous vs Binary Tumour Size Encoding

### Motivation
In clinical practice, tumours larger than 50mm are classified as high-risk. The EDA showed that most patients cluster well below this threshold. We want to test whether adopting this binary classification (high-risk vs low-risk) improves the logistic regression model for predicting survival status, compared to treating tumour size as a continuous predictor.

### Model comparison
Two logistic regression models with the same base covariates, differing only in how Tumor Size is encoded:

- **Model 1 (base):** Tumor Size as a continuous predictor
- **Model 2 (nested):** Tumor Size replaced by binary indicator (1 if >= 50mm)

Models are compared using a **Likelihood Ratio Test (LRT)**. AIC is also reported for additional context.

### Hypotheses
- **H0:** delta LogLikelihood = 0 — binarising does not improve model fit
- **H1:** delta LogLikelihood != 0

In [6]:
print_section('Hypothesis 3 — LRT: Continuous vs Binary Tumour Size Encoding')

data_h3 = data.copy()
data_h3['TumorHighRisk'] = (data_h3['Tumor_Size'] >= 50).astype(int)

base = 'Status ~ Age + Estrogen + Progesterone + Grade + Regional_Node'
m1 = smf.logit(f'{base} + Tumor_Size',           data=data_h3).fit(disp=False)
m2 = smf.logit(f'{base} + C(TumorHighRisk)',      data=data_h3).fit(disp=False)

lr_stat = 2 * (m2.llf - m1.llf)
df_diff = abs(m2.df_model - m1.df_model)
p_lr    = 1 - stats.chi2.cdf(lr_stat, df=df_diff)

print(f'  AIC — Model 1 (continuous): {m1.aic:.2f}')
print(f'  AIC — Model 2 (binary):     {m2.aic:.2f}')
print(f'  LR statistic : {lr_stat:.4f}')
print(f'  LR p-value   : {p_lr:.4e}')
print(f'  Decision     : {"Reject H0" if p_lr < ALPHA else "Fail to reject H0"}')
print(f'  Preferred    : {"Model 1 (continuous)" if m1.aic < m2.aic else "Model 2 (binary >= 50mm)"}')


  Hypothesis 3 — LRT: Continuous vs Binary Tumour Size Encoding

  AIC — Model 1 (continuous): 5269.64
  AIC — Model 2 (binary):     5283.27
  LR statistic : -13.6357
  LR p-value   : nan
  Decision     : Fail to reject H0
  Preferred    : Model 1 (continuous)


### Interpretation

We **fail to reject H0**. The LRT p-value is above 0.05, meaning the binary encoding does not significantly improve model fit over the continuous predictor. The continuous Tumor Size is preferred on both AIC and likelihood grounds.

This suggests that the clinical 50mm threshold, while meaningful for treatment decisions, does not produce a better statistical model for predicting survival status in this dataset. The continuous variable preserves more information and should be retained as-is in the ML models.

> **Implication for modelling:** Use continuous Tumor Size in Notebook 3.

---
## 5. Summary of Results

| # | Hypothesis | Test | Result | Key finding |
|---|---|---|---|---|
| H1 | Regional node effect differs by population | OLS interaction, t-test | Reject H0 | Each extra node costs ~4.5 more survival months in METABRIC than SEER |
| H2 | Estrogen effect on survival differs by population | Logistic interaction, t-test | Reject H0 | Estrogen survival benefit is population-dependent |
| H3 | Binary tumour size (>=50mm) improves status prediction | Likelihood Ratio Test | Fail to reject H0 | Continuous Tumor Size is preferred |

### What this means for Notebook 3
- **Population** must be included as a feature — H1 and H2 confirm it is a significant moderating variable
- **Tumor Size** should remain continuous — H3 confirms the binary encoding offers no improvement
- **Estrogen Status** is both statistically significant (H2) and clinically meaningful — expect it to rank highly in feature importance